# Notebook 3 — SpaceX SQL Analysis

## Step 1 — Importing Libraries and Setting Up SQLite

In this notebook, we will use SQL to analyze historical SpaceX Falcon 9 launch data. We will use Python's built-in SQLite library to create a database, store launch records, and execute SQL queries.

In [1]:
# Step 1: Import libraries

import sqlite3
import pandas as pd

# Create an SQLite database connection
conn = sqlite3.connect("spacex.db")

print("Libraries imported successfully!")
print("SQLite database connection established!")
print("SQLite version:", sqlite3.sqlite_version)

Libraries imported successfully!
SQLite database connection established!
SQLite version: 3.45.1


###Step 2 — Loading the SpaceX Dataset into Pandas
We'll upload the spacex_launches_processed.csv file we downloaded from Notebook 1.

In [2]:
# Step 2: Upload the processed SpaceX dataset

from google.colab import files

uploaded = files.upload()

# Read the uploaded CSV file
spacex_df = pd.read_csv("spacex_launches_processed.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", spacex_df.shape)

print("\nFirst five records:")
display(spacex_df.head())

print("\nColumn names:")
print(spacex_df.columns.tolist())

Saving spacex_launches_processed.csv to spacex_launches_processed.csv
Dataset loaded successfully!
Dataset shape: (90, 18)

First five records:


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,1
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,1
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,1
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,1



Column names:
['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude', 'Class']


###Step 3 — Creating a SQL Table and Importing Records
In this step, we'll transfer our DataFrame into SQLite so we can perform SQL queries on it.

In [3]:
# Step 3: Import SpaceX data into SQLite

spacex_df.to_sql(
    "SPACEX_LAUNCHES",
    conn,
    if_exists="replace",
    index=False
)

print("SpaceX SQL table created successfully!")

# Verify records using SQL
query = """
SELECT COUNT(*) AS TotalLaunches
FROM SPACEX_LAUNCHES;
"""

result = pd.read_sql_query(query, conn)

display(result)

SpaceX SQL table created successfully!


,TotalLaunches
0,90


## Step 4 — Analyzing Launches by Launch Site

In this step, we use SQL aggregation to calculate the number of Falcon 9 launches conducted at each launch site.

In [4]:
# Step 4: Count launches by launch site

query = """
SELECT
    LaunchSite,
    COUNT(*) AS TotalLaunches
FROM SPACEX_LAUNCHES
GROUP BY LaunchSite
ORDER BY TotalLaunches DESC;
"""

launch_site_results = pd.read_sql_query(query, conn)

print("Launches by launch site:")
display(launch_site_results)

Launches by launch site:


,LaunchSite,TotalLaunches
0,CCAFS SLC 40,55
1,KSC LC 39A,22
2,VAFB SLC 4E,13


## Step 5 — Payload Mass Analysis by Launch Site

In this step, we use SQL aggregate functions to calculate the average, minimum, and maximum payload masses for each launch site.

In [5]:
# Step 5: Analyze payload mass by launch site

query = """
SELECT
    LaunchSite,
    COUNT(*) AS TotalLaunches,
    ROUND(AVG(PayloadMass), 2) AS AveragePayloadKg,
    ROUND(MIN(PayloadMass), 2) AS MinimumPayloadKg,
    ROUND(MAX(PayloadMass), 2) AS MaximumPayloadKg
FROM SPACEX_LAUNCHES
GROUP BY LaunchSite
ORDER BY AveragePayloadKg DESC;
"""

payload_results = pd.read_sql_query(query, conn)

print("Payload mass statistics by launch site:")
display(payload_results)

Payload mass statistics by launch site:


,LaunchSite,TotalLaunches,AveragePayloadKg,MinimumPayloadKg,MaximumPayloadKg
0,KSC LC 39A,22,7606.45,2490.0,15600.0
1,VAFB SLC 4E,13,5919.46,475.0,9600.0
2,CCAFS SLC 40,55,5548.21,350.0,15600.0


## Step 6 — Landing Outcome Analysis by Launch Site

In this step, we use SQL aggregation to compare the number and percentage of Class 1 and Class 0 outcomes across different SpaceX launch sites.

In [6]:
# Step 6: Analyze landing outcomes by launch site

query = """
SELECT
    LaunchSite,
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS Class0Count,
    ROUND(AVG(Class) * 100, 2) AS Class1Percentage
FROM SPACEX_LAUNCHES
GROUP BY LaunchSite
ORDER BY Class1Percentage DESC;
"""

landing_results = pd.read_sql_query(query, conn)

print("Landing classification by launch site:")
display(landing_results)

Landing classification by launch site:


,LaunchSite,TotalLaunches,Class1Count,Class0Count,Class1Percentage
0,KSC LC 39A,22,21,1,95.45
1,CCAFS SLC 40,55,47,8,85.45
2,VAFB SLC 4E,13,11,2,84.62


## Step 7 — Analyzing Annual Launch Trends

In this step, we use SQL date functions and aggregation to examine the number of Falcon 9 launches conducted each year between 2010 and 2020.

In [7]:
# Step 7: Analyze annual SpaceX launch trends

query = """
SELECT
    CAST(strftime('%Y', Date) AS INTEGER) AS LaunchYear,
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    ROUND(AVG(Class) * 100, 2) AS Class1Percentage
FROM SPACEX_LAUNCHES
GROUP BY strftime('%Y', Date)
ORDER BY LaunchYear ASC;
"""

yearly_results = pd.read_sql_query(query, conn)

print("Annual SpaceX launch trends:")
display(yearly_results)

Annual SpaceX launch trends:


,LaunchYear,TotalLaunches,Class1Count,Class1Percentage
0,2010,1,1,100.00
1,2012,1,1,100.00
2,2013,3,2,66.67
3,2014,6,5,83.33
4,2015,6,3,50.00
5,2016,8,5,62.50
6,2017,18,18,100.00
7,2018,18,17,94.44
8,2019,10,10,100.00
9,2020,19,17,89.47


## Step 8 — Analyzing Launches by Orbit

In this step, we use SQL aggregation to identify the most frequently targeted orbital categories and compare their Class 1 percentages.

In [8]:
# Step 8: Analyze launches by orbit

query = """
SELECT
    Orbit,
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS Class0Count,
    ROUND(AVG(Class) * 100, 2) AS Class1Percentage
FROM SPACEX_LAUNCHES
GROUP BY Orbit
ORDER BY TotalLaunches DESC;
"""

orbit_results = pd.read_sql_query(query, conn)

print("Launch distribution by orbit:")
display(orbit_results)

Launch distribution by orbit:


,Orbit,TotalLaunches,Class1Count,Class0Count,Class1Percentage
0,GTO,27,25,2,92.59
1,ISS,21,16,5,76.19
2,VLEO,14,12,2,85.71
3,PO,9,7,2,77.78
4,LEO,7,7,0,100.00
5,SSO,5,5,0,100.00
6,MEO,3,3,0,100.00
7,SO,1,1,0,100.00
8,HEO,1,1,0,100.00
9,GEO,1,1,0,100.00


## Step 9 — Analyzing Booster Reusability

In this step, we use SQL aggregation to compare launch counts and Class 1 percentages between previously reused and non-reused Falcon 9 boosters.

In [9]:
# Step 9: Analyze booster reuse and landing classification

query = """
SELECT
    Reused,
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS Class0Count,
    ROUND(AVG(Class) * 100, 2) AS Class1Percentage,
    ROUND(AVG(Flights), 2) AS AverageFlights
FROM SPACEX_LAUNCHES
GROUP BY Reused
ORDER BY Reused;
"""

reuse_results = pd.read_sql_query(query, conn)

print("Booster reuse analysis:")
display(reuse_results)

Booster reuse analysis:


,Reused,TotalLaunches,Class1Count,Class0Count,Class1Percentage,AverageFlights
0,0,53,44,9,83.02,1.00
1,1,37,35,2,94.59,2.92


## Step 10 — Booster Version Analysis

In this step, we use SQL aggregation to compare launch frequency and Class 1 percentages across Falcon 9 booster versions.

In [10]:
# Step 10: Analyze booster versions

query = """
SELECT
    BoosterVersion,
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS Class0Count,
    ROUND(AVG(Class) * 100, 2) AS Class1Percentage
FROM SPACEX_LAUNCHES
GROUP BY BoosterVersion
ORDER BY TotalLaunches DESC;
"""

booster_results = pd.read_sql_query(query, conn)

print("Booster version analysis:")
display(booster_results)

Booster version analysis:


,BoosterVersion,TotalLaunches,Class1Count,Class0Count,Class1Percentage
0,Falcon 9,90,79,11,87.78


## Step 11 — Inspecting Booster Block Information

The BoosterVersion column contains only the value Falcon 9.
We will inspect the Block column to determine whether it provides
more detailed information about booster generations.

In [11]:
# Step 11: Inspect booster block distribution

query = """
SELECT
    Block,
    COUNT(*) AS TotalLaunches
FROM SPACEX_LAUNCHES
GROUP BY Block
ORDER BY Block;
"""

block_results = pd.read_sql_query(query, conn)

print("Booster block distribution:")
display(block_results)

Booster block distribution:


,Block,TotalLaunches
0,1.0,19
1,2.0,6
2,3.0,15
3,4.0,11
4,5.0,39


## Step 12 — Landing Classification by Booster Block

In this step, we compare Class 1 and Class 0 outcomes across
Falcon 9 booster blocks to investigate differences in the
course-defined landing classification.

In [12]:
# Step 12: Analyze landing classification by booster block

query = """
SELECT
    Block,
    COUNT(*) AS TotalLaunches,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS Class0Count,
    ROUND(AVG(Class) * 100, 2) AS Class1Percentage
FROM SPACEX_LAUNCHES
GROUP BY Block
ORDER BY Block;
"""

block_landing_results = pd.read_sql_query(query, conn)

print("Landing classification by booster block:")
display(block_landing_results)

Landing classification by booster block:


,Block,TotalLaunches,Class1Count,Class0Count,Class1Percentage
0,1.0,19,12,7,63.16
1,2.0,6,6,0,100.00
2,3.0,15,14,1,93.33
3,4.0,11,11,0,100.00
4,5.0,39,36,3,92.31


## Step 13 — Final SQL Data Integrity Check

In this step, we verify the total number of records, unique
flight numbers, missing values, and classification counts
to ensure that the SQL dataset is consistent.

In [13]:
# Step 13: Final SQL data validation

query = """
SELECT
    COUNT(*) AS TotalRecords,
    COUNT(DISTINCT FlightNumber) AS UniqueFlights,
    SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS Class1Count,
    SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS Class0Count,
    SUM(CASE WHEN Class IS NULL THEN 1 ELSE 0 END) AS MissingClass,
    SUM(CASE WHEN PayloadMass IS NULL THEN 1 ELSE 0 END) AS MissingPayloadMass
FROM SPACEX_LAUNCHES;
"""

validation_results = pd.read_sql_query(query, conn)

print("Final SQL dataset validation:")
display(validation_results)

Final SQL dataset validation:


,TotalRecords,UniqueFlights,Class1Count,Class0Count,MissingClass,MissingPayloadMass
0,90,90,79,11,0,0
